# 캐럿·컷·색상·투명도로 다이아몬드 가격을 얼마나 설명할 수 있는가

## 1. 분석 개요

### 분석 주제

공개 데이터인 Diamonds 데이터셋을 이용해 **다이아몬드의 특성이 가격과 어떤 관련이 있으며, 가격 차이를 얼마나 설명할 수 있는지** 확인한다. 캐럿·컷·색상·투명도를 중심으로 데이터를 살펴보고, 선형회귀의 전체 과정인 품질 점검 → 탐색적 분석 → 모델링 → 진단 → 해석을 연습한다.

이 분석에서 `price`는 다이아몬드의 가격을 뜻한다.

### 분석 질문

- **Q1.** 캐럿·컷·색상·투명도를 비롯한 다이아몬드 특성 중 가격과 가장 뚜렷한 관련이 있는 요인은 무엇인가?
- **Q2.** 캐럿이 비슷한 다이아몬드라도 컷·색상·투명도에 따라 가격 차이가 나타나는가?
- **Q3.** 기본 선형회귀와 가격의 로그 변환·캐럿의 제곱항·캐럿과 품질 특성의 상호작용을 반영한 모형을 비교했을 때, 어떤 모형이 다이아몬드 가격을 더 잘 설명하고 예측하는가?

**종속변수:** `price` — 다이아몬드 가격(연속형, 달러 단위) → 회귀 문제

**설명변수:** `carat`, `cut`, `color`, `clarity`, `depth`, `table`, `x`, `y`, `z`

### 이번 노트북에서 연습할 기법 — 선형회귀

데이터의 구조와 품질을 점검하고, 탐색적 분석을 통해 각 변수의 분포와 변수 간 관계를 확인한다. 이후 여러 설명변수를 함께 사용하는 다중선형회귀를 적용하여 다른 조건이 같을 때 각 다이아몬드 특성이 가격과 어떤 관계를 보이는지 살펴본다.

구체적인 자료형 변환, 변수 변환 및 모형 구성 방법은 데이터의 실제 분포와 탐색적 분석 결과를 확인한 뒤 결정한다.

#### 1-1. 분석 환경 설정

> 분석에 필요한 개별 함수 **import**

In [1]:
import sys

portfolio_path = r"C:\Data-Analysis-Portfolio"

if portfolio_path not in sys.path:
    sys.path.insert(0, portfolio_path)

from jihyelab import upload_data

In [2]:
from helpers import *
from IPython.display import display, Markdown

import pandas as pd
import numpy as np
import statsmodels.api as sm

---
## #2. 데이터 소개

#### 2-1. 데이터 로딩

In [7]:
# 캐글데이터 주소 모음집
data = upload_data(
    "Diamonds",
    local=r"C:\Data-Analysis-Portfolio\dataset_repository",
)

data = data.drop(columns=["Unnamed: 0"])

data.head()

📚 다이아몬드의 중량·커팅 품질·색상·투명도·크기 등의 특성을 바탕으로 가격을 분석하고 예측하는 데이터셋 (출처: https://www.kaggle.com/datasets/shivam2503/diamonds)

    field    description
--  -------  ------------------------------------------------------------
 0  carat    다이아몬드 중량(캐럿)
 1  cut      다이아몬드 커팅 품질 (Fair, Good, Very Good, Premium, Ideal)
 2  color    다이아몬드 색상 등급 (J부터 D까지이며 D가 가장 높은 등급)
 3  clarity  다이아몬드 투명도 등급
 4  depth    다이아몬드 전체 깊이 비율
 5  table    다이아몬드 상단면의 너비 비율
 6  price    다이아몬드 가격(미국 달러)
 7  x        다이아몬드 길이(mm)
 8  y        다이아몬드 너비(mm)
 9  z        다이아몬드 깊이(mm)



,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


#### 문자열 공백 제거
- 문자열형 변수의 앞뒤 불필요한 공백을 제거해, 같은 값이 공백 때문에 서로 다른 범주로 인식되는 오류를 방지하기 위해 사용한다.

In [9]:
for col in data.select_dtypes(include=['object', 'category']).columns:
    data[col] = data[col].astype(str).str.strip()

#### 2-2. 출처와 분석 단위

- **출처:** Diamonds — Kaggle 배포본 [`shivam2503/diamonds`], 다이아몬드의 가격과 품질·물리적 특성을 포함한 데이터셋
- **수집 기간:** 배포 자료에 구체적인 데이터 수집 연도 및 기간이 명시되어 있지 않음
- **데이터 규모:** 총 53,940개의 다이아몬드, 10개 변수
- **분석 단위:** 1행 = 개별 다이아몬드 1개. 각 행에는 캐럿(`carat`), 컷 등급(`cut`), 색상 등급(`color`), 투명도 등급(`clarity`), 깊이 비율(`depth`), 테이블 비율(`table`), 가격(`price`), 길이·너비·높이(`x`, `y`, `z`)가 기록되어 있음

#### 2-3. 변수 정의서

| 변수명 | 의미 | 역할 | 척도 | 단위 |
|---|---|---|---|---|
| `price` | 다이아몬드의 판매 가격 | 종속변수 | 연속형 | 달러($) |
| `carat` | 다이아몬드의 무게 | 독립변수 | 연속형 | 캐럿(ct) |
| `cut` | 다이아몬드의 컷 품질 등급 | 독립변수 | 순서형(5등급) | Fair < Good < Very Good < Premium < Ideal |
| `color` | 다이아몬드의 색상 등급 | 독립변수 | 순서형(7등급) | J < I < H < G < F < E < D |
| `clarity` | 다이아몬드의 투명도 등급 | 독립변수 | 순서형(8등급) | I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF |
| `depth` | 다이아몬드의 전체 깊이 비율 | 독립변수 | 연속형 | % |
| `table` | 다이아몬드 상단면의 너비 비율 | 독립변수 | 연속형 | % |
| `x` | 다이아몬드의 길이 | 독립변수 | 연속형 | mm |
| `y` | 다이아몬드의 너비 | 독립변수 | 연속형 | mm |
| `z` | 다이아몬드의 깊이(높이) | 독립변수 | 연속형 | mm |

※ 전체 53,940행, 10개 분석 변수로 구성된다. `price`, `carat`, `depth`, `table`, `x`, `y`, `z`는 수치형 변수이며, `cut`, `color`, `clarity`는 등급 간 순서가 존재하는 범주형 변수이다. 이번 분석에서는 다이아몬드 가격인 `price`를 종속변수로 설정하고 나머지 변수를 독립변수로 사용한다. `cut`, `color`, `clarity`는 범주형 변수이므로 회귀모형에 투입할 때 적절한 인코딩 과정을 거친다.

#### 2-4. 초기 가설

- **H1. 캐럿(`carat`)이 클수록 다이아몬드 가격(`price`)이 높아질 것이다.**
- **H2. 컷(`cut`), 색상(`color`), 투명도(`clarity`) 등 품질 등급에 따라 다이아몬드 가격에 유의한 차이가 있을 것이다.**
- **H3. 다이아몬드의 길이(`x`), 너비(`y`), 깊이(`z`)가 클수록 가격(`price`)이 높아질 것이다.**

위 가설은 분석 전의 예상이며, 실제 데이터와 통계적 검정 및 모형 진단 결과에 따라 기각하거나 수정한다.

#### 2-5. 데이터의 한계

- **인과관계를 확인할 수 없다.** 다이아몬드의 특성과 가격 간의 관련성을 분석하는 관찰 자료이므로, 특정 특성이 가격 상승을 직접 유발한다고 단정할 수 없다.
- **가격에 영향을 주는 모든 정보가 포함되어 있지 않다.** 브랜드, 판매처, 감정 기관, 원산지, 시장 상황, 세공 방식 등의 정보가 없어 실제 다이아몬드 가격 차이를 완전히 설명하기 어렵다.
- **데이터의 수집 시점과 방식이 명확하지 않다.** 구체적인 수집 연도와 거래 시점이 공개되어 있지 않아 현재의 다이아몬드 시장 가격에 그대로 적용하기 어렵다.
- **특정 시장이나 표본의 특성이 반영되었을 가능성이 있다.** 데이터가 전체 다이아몬드 시장을 대표한다고 보기 어려우므로 분석 결과를 모든 다이아몬드 거래에 일반화할 때 주의해야 한다.
- **극단적인 크기나 가격의 관측값이 분석 결과에 영향을 줄 수 있다.** 높은 가격이나 큰 캐럿 값을 단순히 이상치로 판단해 제거하지 않고, 데이터 오류 여부와 모형에 미치는 영향을 함께 확인해야 한다.
- **물리적 크기 변수 간 강한 상관관계가 존재할 수 있다.** `carat`, `x`, `y`, `z`는 모두 다이아몬드의 크기와 관련된 변수이므로 다중공선성이 발생할 가능성이 있으며, 회귀분석 전에 상관관계와 VIF 등을 확인할 필요가 있다.
- **실제 다이아몬드 감정이나 가격 책정 모형은 아니다.** 이 노트북은 다이아몬드의 가격과 특성 간 관계를 이해하고 통계 분석 및 회귀분석을 연습하기 위한 것으로, 실제 거래가격이나 감정가를 결정하는 근거로 사용하지 않는다.